# CascadeLUT Demo — ToyADMOS (ToyCar anomaly detection)

Demonstrates the full late-input cascade pipeline on ToyADMOS:
1. **Phase 1** — dense training with full input to learn feature importance ordering
2. **Phase 2** — cascade dense training to learn sparse connectivity masks
3. **Phase 3** — sparse incremental training with learned masks
4. **neq2lut** — convert NEQs to LUT truth tables
5. **Verilog** — generate synthesisable cascade RTL

Epoch counts are reduced from the paper run (15/15/600) to (3/5/15) so the notebook completes quickly.

Data must be pre-processed and stored in `./data/train.pkl` and `./data/test.pkl` (run the dataset preparation script first if not present).

In [32]:
import sys, os, types

import cascadelut
import cascadelut.nn
import cascadelut.quant
import cascadelut.synthesis

import wandb
wandb.init(mode='disabled')  # suppress wandb calls inside train.py

In [33]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from cascadelut.nn import generate_truth_tables, lut_inference, module_list_to_verilog_module
from models import ToyCarCascadeModel, _parse_cascade_chunks
from dataset import load_dataset
import train as _train

## Config

Same architecture as the `"toycar"` config in `train.py`. Only epoch counts are reduced.

In [34]:
CUDA = torch.cuda.is_available()
print(f'CUDA: {CUDA}')

demo_cfg = {
    'hidden_layers':  [500, 512, 128, 32, 8],
    'support_layers': [0, 0, 1, 1, 1, 1],
    'cascade_layers': [1, 2, 0, 0, 0],
    'input_bitwidth':   2, 'hidden_bitwidth':  2,
    'output_bitwidth':  4, 'support_bitwidth': 2,
    'input_fanin':   4, 'hidden_fanin':   4,
    'output_fanin':  6, 'support_fanin':  4,
    'cascade_fanin': 4, 'width_n': 32, # reduced for demo
    'batch_size': 256, 'learning_rate': 0.0016,
    'weight_decay': 0.0002, 'label_smoothing': 0.05,
    'input_dropout': 0.08, 'features_per_cycle': 112,
    'dense1_epochs': 5, 'dense2_epochs': 10, 'epochs': 25, # reduced for demo
    'warmup_epochs': 2,
    'wd': 0.00012, 'wd1': 0.03,
    'start_temp': 1.8, 'end_temp': 0.02,
    'anneal_epochs': 15, 'use_anneal': True,
    'threshold_reg_strength': 0.000005,
    'threshold_reg_noise_std': 0.0,
    'use_augmentation': False,
    'seed': 42, 'cuda': CUDA,
}

CUDA: True


## Load dataset

The ToyADMOS toy-car dataset is stored as pre-processed pickle files. `input_length` and `output_length` are derived from the data.

In [35]:
X_train, y_train, X_test, y_test = load_dataset('./data/')

input_length  = X_train.shape[1]
output_length = len(set(y_train.tolist()))
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'input_length={input_length}, output_length={output_length}')

demo_cfg['input_length']  = input_length
demo_cfg['output_length'] = output_length

# Build a test loader for standalone evaluation
X_test_t = torch.from_numpy(X_test).float()
y_test_t  = torch.from_numpy(y_test.astype(np.int64)).long()
test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=demo_cfg['batch_size'], shuffle=False,
)

Train: (10000, 640), Test: (4000, 640)
input_length=640, output_length=2


In [36]:
def eval_model(model, loader, cuda=True):
    """Run test-set evaluation without touching wandb or global state."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            if cuda:
                data, target = data.cuda(), target.cuda()
            correct += model(data).argmax(1).eq(target).sum()
    return 100.0 * correct / len(loader.dataset)

## Phase 1: Dense training with full input

The model sees the complete feature vector at every cascade layer (`full_input_mode=True`). After training, we rank features by weight magnitude to determine per-chunk importance ordering.

In [37]:
import random
seed = demo_cfg['seed']
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
if CUDA:
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

model_cfg = dict(demo_cfg)
model_cfg['dense_forward']    = True
model_cfg['full_input_mode']  = True
model_cfg['imask']            = torch.tensor([]).cuda() if CUDA else torch.tensor([])
model_cfg['importance_order'] = torch.arange(input_length, dtype=torch.long)

# train_cfg / opts mirror train.py's __main__ setup
train_cfg = {k: demo_cfg[k] for k in _train.training_config if k in demo_cfg}
opts      = {'cuda': CUDA, 'log_dir': 'cascade_demo', 'checkpoint': None, 'device': 0}

os.makedirs('test_cascade_demo', exist_ok=True)

# Expose globals that train.py functions read
_train.model_cfg = model_cfg
_train.train_cfg = train_cfg

model_phase1 = ToyCarCascadeModel(model_cfg)
if CUDA:
    model_phase1.cuda()

print('=== Phase 1: Dense (full-input) ===')
_train.train(model_phase1, train_cfg, opts)

=== Phase 1: Dense (full-input) ===
Epoch: 0/5	Test Acc: 69.53
Epoch: 1/5	Test Acc: 76.65
Epoch: 2/5	Test Acc: 65.60
Epoch: 3/5	Test Acc: 78.22
Epoch: 4/5	Test Acc: 69.00


## Compute importance ordering

Rank the input features by how heavily the dense layers weight them, then partition into non-overlapping chunks of `bits_per_cycle` features.

In [38]:
model_phase1.eval()
importance_order = _train.calculate_cascade_importance_order_frequency(
    model_phase1, model_cfg, CUDA
)
print(f'Importance order shape: {importance_order.shape}')

B = demo_cfg['features_per_cycle']
cascade_chunks_vis = _parse_cascade_chunks(demo_cfg['cascade_layers'])
all_used_vis = sorted(set(c for lc in cascade_chunks_vis for c in lc))
for c in all_used_vis:
    seg = importance_order[c * B : (c + 1) * B].tolist()
    print(f'  Chunk {c}: top-5 feature indices = {seg[:5]} ...')

torch.save(importance_order, 'test_cascade_demo/importance_order.pth')

# Update config for subsequent phases
model_cfg['importance_order'] = importance_order
model_cfg['full_input_mode']  = False
_train.model_cfg = model_cfg

del model_phase1
if CUDA:
    torch.cuda.empty_cache()

Importance order shape: torch.Size([640])
  Chunk 0: top-5 feature indices = [467, 639, 39, 340, 468] ...
  Chunk 1: top-5 feature indices = [466, 630, 3, 71, 168] ...


## Phase 2: Cascade dense training → sparse mask

Train the cascade architecture in dense mode with chunks arriving in importance order. After training, extract `imask`: the learned sparse connectivity for each neuron.

In [39]:
model_cfg['dense_forward'] = True
_train.model_cfg = model_cfg

model_phase2 = ToyCarCascadeModel(model_cfg)
if CUDA:
    model_phase2.cuda()

print('=== Phase 2: Cascade dense ===')
_train.train(model_phase2, train_cfg, opts)

imask = _train.GetInputMask(model_phase2, CUDA)
print(f'Extracted imask: {imask.shape}')
torch.save(imask, 'test_cascade_demo/imask_cascade.pth')

del model_phase2
if CUDA:
    torch.cuda.empty_cache()

=== Phase 2: Cascade dense ===
Epoch: 0/10	Test Acc: 75.42
Epoch: 1/10	Test Acc: 74.45
Epoch: 2/10	Test Acc: 70.85
Epoch: 3/10	Test Acc: 70.95
Epoch: 4/10	Test Acc: 76.88
Epoch: 5/10	Test Acc: 67.88
Epoch: 6/10	Test Acc: 69.10
Epoch: 7/10	Test Acc: 75.80
Epoch: 8/10	Test Acc: 69.92
Epoch: 9/10	Test Acc: 78.05
Extracted imask: torch.Size([1012, 6])


## Phase 3: Sparse incremental training

Now train with the learned `imask` and real chunked input loading. Each cascade layer only receives the features its neurons were connected to during the dense phase.

In [40]:
model_cfg['dense_forward'] = False
model_cfg['imask']         = imask.cuda() if CUDA else imask
_train.model_cfg = model_cfg

model = ToyCarCascadeModel(model_cfg)
if CUDA:
    model.cuda()

print('=== Phase 3: Sparse incremental ===')
_train.train(model, train_cfg, opts)
print(f'Final test accuracy: {eval_model(model, test_loader, CUDA):.2f}%')

=== Phase 3: Sparse incremental ===
Final test accuracy: 75.73%


## NEQ → LUT conversion

Calculate the Boolean truth table for every neuron, then switch the model to LUT-based forward inference.

In [41]:
print('Converting NEQs to LUTs...')
generate_truth_tables(model, verbose=True)
lut_inference(model)
print(f'LUT model accuracy: {eval_model(model, test_loader, CUDA):.2f}%')

Converting NEQs to LUTs...
Fixed input_quant for layer 5 from previous layer's output_quant
Patched cascade layer module_list.1
Calculating truth tables for module_list.0
Truth tables generated for 500 neurons
Calculating truth tables for module_list.1
Truth tables generated for 512 neurons
Calculating truth tables for module_list.2
Truth tables generated for 128 neurons
Calculating truth tables for module_list.3
Truth tables generated for 32 neurons
Calculating truth tables for module_list.4
Truth tables generated for 8 neurons
Calculating truth tables for module_list.5
Truth tables generated for 2 neurons
LUT model accuracy: 75.65%


## Verilog generation

Generate per-layer Verilog modules and the cascade top-level that wires them together with pipeline registers and chunk-loading control logic.

In [42]:
verilog_dir = 'test_cascade_demo/verilog'
os.makedirs(verilog_dir, exist_ok=True)

cascade_chunks = _parse_cascade_chunks(demo_cfg['cascade_layers'])

module_list_to_verilog_module(
    model.module_list,
    'cascade_demo',
    verilog_dir,
    add_registers=True,
    cascade_config={
        'enabled':        True,
        'bits_per_cycle': demo_cfg['features_per_cycle'],
        'cascade_chunks': cascade_chunks,
        'cascade_layers': demo_cfg['cascade_layers'],
    },
)
print(f'Verilog written to: {verilog_dir}/cascade_demo.v')
print(f'Layer files: {[f for f in os.listdir(verilog_dir) if f.endswith(".v")]}')

Generated cascade top module: test_cascade_demo/verilog/cascade_demo.v
Verilog written to: test_cascade_demo/verilog/cascade_demo.v
Layer files: ['layer0_N282.v', 'layer0_N391.v', 'layer1_N473.v', 'layer2_N40.v', 'layer0_N62.v', 'layer1_N73.v', 'layer0_N447.v', 'layer1_N183.v', 'layer2_N18.v', 'layer0_N118.v', 'layer0_N1.v', 'layer2_N4.v', 'layer1_N174.v', 'layer1_N484.v', 'layer0_N140.v', 'layer1_N84.v', 'layer0_N95.v', 'layer1_N241.v', 'layer0_N366.v', 'layer0_N275.v', 'layer1_N219.v', 'layer1_N352.v', 'layer1_N297.v', 'layer1_N384.v', 'layer2_N39.v', 'layer3_N28.v', 'layer0_N466.v', 'layer0_N196.v', 'layer1_N452.v', 'layer0_N43.v', 'layer1_N52.v', 'layer2_N61.v', 'layer0_N161.v', 'layer2_N96.v', 'layer1_N155.v', 'layer0_N491.v', 'layer0_N139.v', 'layer1_N238.v', 'layer1_N373.v', 'layer2_N109.v', 'layer0_N254.v', 'layer0_N347.v', 'layer1_N260.v', 'layer0_N405.v', 'layer1_N69.v', 'layer0_N78.v', 'layer1_N469.v', 'layer3_N13.v', 'layer1_N31.v', 'layer0_N20.v', 'layer1_N431.v', 'layer1_